In [101]:
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, YEARS, RESULTS_DIR, SRC_DIR, SRC_DIRS, CHOC_SCENARIO

CONTENT = "dir"        # "dir" : rd_dir_CI | "tot" : rd_tot_CI
YEAR = 2022 # YEARS[0]       # année de la table de base

# Blocs du protocole T4.2. Les pays absents de ce dictionnaire vont dans DEFAULT_BLOCK ("RW" regroupe donc, ici, le reste du monde, l'Asie hors Singapour / Taïwan et l'Afrique du Sud : à affiner si la base distingue ces régions dans agg_base_to_rd.xlsx).
EU27 = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT",
        "RO", "SK", "SI", "ES", "SE"]
BLOCKS = {
    "USA": ["US"],
    "EU27": EU27,
    "RestOECD": ["AU", "CA", "JP", "KR", "MX", "NZ", "NO", "CH", "UK"],
    "China": ["CN"],
    "RestBRICS": ["BR", "RU", "IN", "ZA"],
    "RestAsia": ["BD", "KH", "ID", "LA", "MY", "MM", "NP", "PK", "PH", "SG", "TW", "TJ", "TL", "TH", "VN"],
}
REGION_TO_BLOCK = {region: block for block, regions in BLOCKS.items() for region in regions}
DEFAULT_BLOCK = "RoW"

# Secteurs communs manufacturiers du protocole (Table 1 : textiles ... manufactures nec ; hors agroalimentaire et produits pétroliers)
MANUFACTURING = ["C13_C15", "C16_C18", "C20+C21", "C22", "C23", "C24", "C25", "C26", "C27", "C28", "C29", "C30", "C31_C33"]
NONMANUF = "NONMANUF"   # intrants non manufacturiers : jamais choqués, ils restent dans les stocks

In [ ]:
def read_matrix(path):
    """Fichier du programme principal : matrice (Region, Level, Item) en lignes et en colonnes."""
    return pd.read_csv(path, sep=";", header=[0, 1, 2], index_col=[0, 1, 2])


def flat(m):
    """(Region, Level, Item) -> (Region, Sector), pour les calculs."""
    m = m.copy()
    m.index, m.columns = m.index.droplevel(1), m.columns.droplevel(1)
    m.index.names = m.columns.names = ["Region", "Sector"]
    return m


# Composantes du stock de connaissance de chaque modèle de l'estimation : (même secteur ?, même pays ?), None = indifférent
#   (2) D = fournisseur du même pays | F = fournisseur étranger
#   (3) I = fournisseur du même secteur que l'acheteur | X = autres secteurs
#   (4) croisement secteur identique ou non x pays identique ou non (même découpage que compute_4_blocks de calc_blocs2.0)
# Les β viennent de choc_tfp/inputs/beta.csv (modèle 2), beta_modele3.csv et beta_modele4.csv.
MODEL_BLOCKS = {
    2: {"D": (None, True), "F": (None, False)},
    3: {"I": (True, None), "X": (False, None)},
    4: {"same_same": (True, True), "same_diff": (True, False), "diff_same": (False, True), "diff_diff": (False, False)},
}


def block_matrices(m, region_to_block=REGION_TO_BLOCK, default_block=DEFAULT_BLOCK):
    """Agrégation en amont (section 3 de la méthodologie) : m = flux de contenu c ou volumes z, lignes = fournisseurs (pays, secteur),
    colonnes = acheteurs (pays, secteur). Chaque flux est classé domestique (même pays) ou étranger (pays différents) AU NIVEAU PAYS,
    puis les deux matrices sont sommées par (bloc, secteur) en lignes et en colonnes. Retourne (domestique, étranger).
    Les flux entre deux pays d'un même bloc restent dans la matrice étrangère (fournisseur = acheteur = ce bloc)."""
    reg_i, reg_j = m.index.get_level_values("Region"), m.columns.get_level_values("Region")
    foreign = np.asarray(reg_i)[:, None] != np.asarray(reg_j)[None, :]
    rows = pd.MultiIndex.from_arrays([[region_to_block.get(r, default_block) for r in reg_i], m.index.get_level_values("Sector")], names=["Bloc", "Sector"])
    cols = pd.MultiIndex.from_arrays([[region_to_block.get(r, default_block) for r in reg_j], m.columns.get_level_values("Sector")], names=["Bloc", "Sector"])
    out = []
    for mask in (~foreign, foreign):
        a = pd.DataFrame(m.to_numpy() * mask, index=rows, columns=cols)
        out.append(a.groupby(level=["Bloc", "Sector"]).sum().T.groupby(level=["Bloc", "Sector"]).sum().T)
    return out


def omega_blocks(index, columns, omega, manufacturing=MANUFACTURING):
    """Facteur de choc omega de chaque flux entre blocs (lignes = (bloc fournisseur, secteur), colonnes = (bloc acheteur, secteur)).
    omega vaut 1 pour les intrants non manufacturiers et les couples non listés ; "ALL" d'abord, les secteurs précis l'écrasent.
    Il ne sert que pour la matrice étrangère : les flux domestiques ne sont jamais choqués."""
    b_i, sec_i = np.asarray(index.get_level_values("Bloc")), np.asarray(index.get_level_values("Sector"))
    b_j = np.asarray(columns.get_level_values("Bloc"))
    om = np.ones((len(index), len(columns)))
    for r in omega.sort_values("sector", key=lambda s: s != "ALL").itertuples():
        rows = (b_i == r.origin) & ((sec_i == r.sector) | (r.sector == "ALL"))
        om[np.ix_(rows, b_j == r.dest)] = 1 + r.delta_pct / 100
    return np.where(np.isin(sec_i, manufacturing)[:, None], om, 1.0)


def shock_blocks(cd, cf, zd, zf, omega, beta, model, manufacturing=MANUFACTURING):
    """Choc sur les stocks de connaissance de chaque (bloc acheteur, secteur), pour le modèle (2), (3) ou (4) : S_k, dlnS_k pour chaque
    composante k de MODEL_BLOCKS[model] et dlnCost = somme des beta_k * dlnS_k.
    cd, cf : contenu domestique / étranger agrégé par block_matrices ; zd, zf : volumes correspondants (None : seul le scénario A est calculé).
    Scénario A : les flux étrangers sont multipliés par omega, le domestique ne bouge pas.
    Scénario B : le volume perdu P = somme (1 - omega) * z^F est comblé par la production domestique du même produit, à l'intensité
    moyenne du bloc rho^D = c^D / z^D (plafonné pour ne pas descendre sous zéro) ; le gain va dans les composantes "même pays" :
    "même secteur" s'il vient du secteur de l'acheteur, "autres secteurs" sinon."""
    idx, cols = cd.index, cd.columns
    bloc_j, sec_j = cols.get_level_values("Bloc"), np.asarray(cols.get_level_values("Sector"))
    same_sec = np.asarray(idx.get_level_values("Sector"))[:, None] == sec_j[None, :]
    om = omega_blocks(idx, cols, omega, manufacturing)
    cdv, cfv = cd.to_numpy(), cf.to_numpy()
    s0, s1 = {}, {}
    for k, (sec, cty) in MODEL_BLOCKS[model].items():
        ms = np.ones_like(same_sec) if sec is None else same_sec == sec
        d = (cdv * ms).sum(0) if cty is not False else 0.0                     # partie domestique (jamais choquée)
        s0[k] = d + ((cfv * ms).sum(0) if cty is not True else 0.0)
        s1[k] = d + ((cfv * om * ms).sum(0) if cty is not True else 0.0)

    none = np.zeros(len(sec_j))
    cases = [("A", none, none)]
    if zd is not None:                                                          # scénario B : a besoin des volumes z
        by_sector = lambda a: pd.DataFrame(a, index=idx, columns=cols).groupby(level="Sector").sum()
        c_dom, z_dom = by_sector(cdv), by_sector(zd.to_numpy())
        lost = by_sector((1 - om) * zf.to_numpy())                              # volume d'intrants étrangers perdus (secteur fournisseur x acheteur)
        rho_d = (c_dom / z_dom.where(z_dom > 0)).fillna(0.0)                    # intensité en R&D du domestique, moyenne du bloc
        gain = np.maximum(rho_d * lost, -c_dom)
        g_same = gain.to_numpy()[gain.index.get_indexer(sec_j), np.arange(len(sec_j))]   # gain venu du secteur de l'acheteur
        cases.append(("B", g_same, gain.sum().to_numpy() - g_same))

    out = []
    for scenario, g_same, g_diff in cases:
        res = pd.DataFrame({"bloc": bloc_j, "sector": sec_j})
        for k, (sec, cty) in MODEL_BLOCKS[model].items():
            g = 0.0 if cty is False else (g_same + g_diff if sec is None else g_same if sec else g_diff)
            with np.errstate(all="ignore"):
                res[f"S_{k}"], res[f"dlnS_{k}"] = s0[k], np.where(s0[k] > 0, np.log((s1[k] + g) / s0[k]), 0.0)
        out.append(res.assign(scenario=scenario))
    res = pd.concat(out, ignore_index=True)
    if beta is not None:
        res = res.merge(beta, on="sector", how="left")
        res["dlnCost"] = sum(res[f"beta_{k}"] * res[f"dlnS_{k}"] for k in MODEL_BLOCKS[model])
    return res


def add_sector_C(res, components, beta=None):
    """Ajoute le secteur agrégé C (total des sous-secteurs C10_C12 ... C31_C33), construit comme dans création_panel_F2.0 :
    pour chaque bloc acheteur et scénario, les stocks avant et après choc (S et S * exp(dlnS)) sont sommés sur les sous-secteurs C*,
    puis dlnS_k = ln(S'_k / S_k) est recalculé sur le total et dlnCost = somme des beta_k * dlnS_k avec les beta du secteur C."""
    sub = res[res["sector"].str.startswith("C")]
    keys = [sub["bloc"], sub["scenario"]]
    agg = pd.DataFrame(index=sub.groupby(keys, sort=False).size().index)
    for k in components:
        s0 = sub[f"S_{k}"].groupby(keys, sort=False).sum()
        s1 = (sub[f"S_{k}"] * np.exp(sub[f"dlnS_{k}"])).groupby(keys, sort=False).sum()      # stock après choc
        with np.errstate(all="ignore"):
            agg[f"S_{k}"], agg[f"dlnS_{k}"] = s0, np.where(s0 > 0, np.log(s1 / s0), 0.0)
    agg = agg.reset_index().assign(sector="C")
    if beta is not None:
        agg = agg.merge(beta, on="sector", how="left")
        agg["dlnCost"] = sum(agg[f"beta_{k}"] * agg[f"dlnS_{k}"] for k in components)
    return pd.concat([res, agg[res.columns]], ignore_index=True)


def sum_sectors(res, components, beta=None):
    """Même tableau avec les blocs sommés : une ligne par (secteur, scénario) au lieu de (bloc, secteur).
    Les stocks avant et après choc (S et S * exp(dlnS)) sont sommés sur les blocs, puis dlnS_k = ln(S'_k / S_k) est recalculé sur les totaux ;
    dlnCost = somme des beta_k * dlnS_k avec les beta du secteur."""
    keys = [res["sector"], res["scenario"]]
    out = pd.DataFrame(index=res.groupby(keys, sort=False).size().index)
    for k in components:
        s0 = res[f"S_{k}"].groupby(keys, sort=False).sum()
        s1 = (res[f"S_{k}"] * np.exp(res[f"dlnS_{k}"])).groupby(keys, sort=False).sum()      # stock après choc
        with np.errstate(all="ignore"):
            out[f"S_{k}"], out[f"dlnS_{k}"] = s0, np.where(s0 > 0, np.log(s1 / s0), 0.0)
    out = out.reset_index()
    out = out[["sector"] + [c_ for k in components for c_ in (f"S_{k}", f"dlnS_{k}")] + ["scenario"]]
    if beta is not None:
        out = out.merge(beta, on="sector", how="left")
        out["dlnCost"] = sum(out[f"beta_{k}"] * out[f"dlnS_{k}"] for k in components)
    return out


In [ ]:
# Test à la main : 2 pays X1, X2 dans le bloc X, 1 pays Y dans le bloc Y, un seul secteur manufacturier "m".
toy_blocks = {"X1": "X", "X2": "X", "Y": "Y"}
idx = pd.MultiIndex.from_product([["X1", "X2", "Y"], ["m"]], names=["Region", "Sector"])
z_toy = pd.DataFrame([[30.0, 10.0, 20.0], [20.0, 40.0, 10.0], [50.0, 30.0, 60.0]], index=idx, columns=idx)   # lignes = fournisseurs, colonnes = acheteurs
c_toy = z_toy.multiply([0.10, 0.20, 0.30], axis=0)                                                              # c = rho (fournisseur) x Z
omega_toy = pd.DataFrame([{"origin": "Y", "dest": "X", "sector": "m", "delta_pct": -25.0}])
beta_toy = pd.DataFrame([{"sector": "m", "beta_D": 0.05, "beta_F": 0.08}])

cd_t, cf_t = block_matrices(c_toy, toy_blocks)
zd_t, zf_t = block_matrices(z_toy, toy_blocks)
t = shock_blocks(cd_t, cf_t, zd_t, zf_t, omega_toy, beta_toy, 2, ["m"]).set_index(["scenario", "bloc"])
# Bloc X : S_D = 3 + 8 = 11 ; S_F = 4 + 1 + 15 + 9 = 29, 23 après choc (les flux de Y x 0.75, X1 <-> X2 inchangés)
#   A : dlnS_F = ln(23/29) ; B : P = 0.25 x (50 + 30) = 20 comblé à rho_D = 11 / 70 -> S_D' = 11 + 20 x 11 / 70, dlnS_D = ln(S_D' / 11)
expected = {("A", "X"): (-0.231802, 0.0, -0.018544), ("B", "X"): (-0.231802, 0.251314, -0.005978),
            ("A", "Y"): (0.0, 0.0, 0.0), ("B", "Y"): (0.0, 0.0, 0.0)}
print(t[["S_D", "S_F", "dlnS_F", "dlnS_D", "dlnCost"]].round(5))
assert all(abs(t.loc[k, ["dlnS_F", "dlnS_D", "dlnCost"]].astype(float).to_numpy() - np.array(v)).max() < 1e-4 for k, v in expected.items())
print("TEST OK")

tA = shock_blocks(cd_t, cf_t, None, None, omega_toy, beta_toy, 2, ["m"])   # scénario A sans z : même résultat que A avec z, pas de ligne B
assert set(tA["scenario"]) == {"A"} and np.allclose(tA.sort_values("bloc")[["dlnS_F", "dlnCost"]].to_numpy(), t.loc["A"].sort_index()[["dlnS_F", "dlnCost"]].to_numpy())

# Cohérence des trois modèles : après le choc, D + F = I + X = les 4 blocs (même stock total), sur un toy à 2 secteurs
idx2 = pd.MultiIndex.from_product([["X1", "X2", "Y"], ["m", "n"]], names=["Region", "Sector"])
z2 = pd.DataFrame((np.arange(36).reshape(6, 6) * 7 % 11) + 1.0, index=idx2, columns=idx2)
c2 = z2.multiply(np.linspace(0.05, 0.30, 6), axis=0)
cd2, cf2 = block_matrices(c2, toy_blocks)
zd2, zf2 = [m.reindex(index=cd2.index, columns=cd2.columns).fillna(0.0) for m in block_matrices(z2, toy_blocks)]
post = {}
for model_ in MODEL_BLOCKS:
    r_ = shock_blocks(cd2, cf2, zd2, zf2, omega_toy, None, model_, ["m", "n"]).set_index(["bloc", "sector", "scenario"])
    post[model_] = (sum(r_[f"S_{k}"] * np.exp(r_[f"dlnS_{k}"]) for k in MODEL_BLOCKS[model_]), sum(r_[f"S_{k}"] for k in MODEL_BLOCKS[model_]))
assert all(np.allclose(post[m_][0], post[2][0]) and np.allclose(post[m_][1], post[2][1]) for m_ in (3, 4))
print("COHERENCE MODELES OK")


In [104]:
#if BASE not in ("EXIOBASE", "FIGARO", "OECD_ICIO", "GTAP"):
#    raise ValueError(f"BASE={BASE!r} : le choc utilise les flux Z d'une base MRIO (EXIOBASE, FIGARO, OECD_ICIO ou GTAP)")


Z_REF_YEAR = 2010   # année dont z_CI sert à reconstruire les volumes des années sans z_CI (coefficients techniques fixes)


def mpath(name, year, suffix="_agg"):
    return Path(SRC_DIR) / BASE / f"output_loop_{BASE}_{year}" / f"{name}_{BASE}_{year}{suffix}.csv"


prod = pd.read_csv(Path(SRC_DIR) / BASE / f"output_loop_{BASE}" / f"production_{BASE}_agg.csv", sep=";").set_index(["Region", "Sector"])
_cache = {}


def load_z(c, year):
    """z_CI de l'année : fichier exact s'il existe, sinon reconstruit : rho (R&D du fournisseur par euro vendu) vient des coefficients
    techniques de Z_REF_YEAR (A = Z / x) et de la production de l'année, puis z = c / rho garde la structure de c de l'année.
    Le scénario B et rd_embod en dépendent : approchés pour les années reconstruites. Retourne (z, source)."""
    if mpath("z_CI", year).exists():
        return flat(read_matrix(mpath("z_CI", year))).reindex(index=c.index, columns=c.columns).fillna(0.0), "z_CI exact"
    if "A" not in _cache:
        for d in (SRC_DIR, PATH_WRK + "SRDV (c,s)/", PATH_WRK + "SRDQ (c,s)/"):   # z_CI de l'année de référence, dans n'importe quel dossier de run
            p = Path(d) / BASE / f"output_loop_{BASE}_{Z_REF_YEAR}" / f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv"
            if p.exists():
                z_ref = flat(read_matrix(p)).reindex(index=c.index, columns=c.columns).fillna(0.0)
                x_ref = prod[str(Z_REF_YEAR)].reindex(c.index).fillna(0.0).to_numpy()
                _cache["A"] = z_ref.to_numpy() / np.where(x_ref > 0, x_ref, 1.0)[None, :]
                break
        else:
            raise FileNotFoundError(f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv introuvable : lancer know_calc_loop4.0 pour l'année {Z_REF_YEAR}")
    x = prod[str(year)].reindex(c.index).fillna(0.0).to_numpy()
    z_prox = _cache["A"] * x[None, :]                                    # volumes avec les coefficients techniques de Z_REF_YEAR
    zs, cs = z_prox.sum(axis=1), c.to_numpy().sum(axis=1)
    rho = np.divide(cs, zs, out=np.zeros(len(zs)), where=zs > 0)         # R&D du fournisseur par euro vendu cette année
    z = np.where(rho[:, None] > 0, np.divide(c.to_numpy(), rho[:, None], out=np.zeros(c.shape), where=rho[:, None] > 0), z_prox)   # z = c / rho
    return pd.DataFrame(z, index=c.index, columns=c.columns), f"z reconstruit (c/rho, A {Z_REF_YEAR})"


def find(name):
    for d in SRC_DIRS:
        p = Path(d) / BASE / f"output_loop_{BASE}_{YEAR}" / f"{name}_{BASE}_{YEAR}_agg.csv"
        if p.exists():
            return p
    raise FileNotFoundError(f"{name}_{BASE}_{YEAR}_agg.csv introuvable dans {SRC_DIRS} : lancer know_calc_loop puis aggregate_to_common_mapping")


inputs = Path(PATH_WRK) / "choc_tfp" / "inputs"
omega = pd.read_csv(inputs / "omega.csv")                                              # origin, dest, sector, delta_pct

prefix = {"dir": "rd_dir_CI", "tot": "rd_tot_CI"}[CONTENT]
c0 = flat(read_matrix(find(prefix)))
if CHOC_SCENARIO == "B":                      # seul le scénario B a besoin des volumes z (le A n'en a pas besoin)
    z0, z_src = load_z(c0, YEAR)
else:
    z0, z_src = None, "scénario A : pas de volumes z"
print(f"volumes {YEAR} : {z_src}")

cd, cf = block_matrices(c0)                       # contenu domestique / étranger, agrégé en (bloc, secteur) AVANT le choc
zd, zf = (None, None) if z0 is None else [m.reindex(index=cd.index, columns=cd.columns).fillna(0.0) for m in block_matrices(z0)]
print(f"{cd.shape[0]} (bloc, secteur) fournisseurs x {cd.shape[1]} (bloc, secteur) acheteurs")

out = Path(RESULTS_DIR) / "choc_tfp" / f"{BASE}"
out.mkdir(parents=True, exist_ok=True)


volumes 2022 : z_CI exact
238 (bloc, secteur) fournisseurs x 238 (bloc, secteur) acheteurs


In [ ]:
# Les trois modèles de l'estimation (voir MODEL_BLOCKS) : un fichier par modèle, une ligne par (bloc acheteur, secteur, scénario),
# plus le secteur agrégé C, et le même tableau avec les blocs sommés (une ligne par secteur et scénario).
results = {}
for model in MODEL_BLOCKS:
    f_beta = inputs / ("beta.csv" if model == 2 else f"beta_modele{model}.csv")
    beta_m = pd.read_csv(f_beta) if f_beta.exists() else None
    res_m = add_sector_C(shock_blocks(cd, cf, zd, zf, omega, beta_m, model), list(MODEL_BLOCKS[model]), beta_m)
    res_m.to_csv(out / f"resultats{model}_blocs_{BASE}_{CONTENT}_{YEAR}_choc{CHOC_SCENARIO}.csv", index=False)
    res_s = sum_sectors(res_m, list(MODEL_BLOCKS[model]), beta_m)
    res_s.to_csv(out / f"resultats{model}_blocs_secteurs_{BASE}_{CONTENT}_{YEAR}_choc{CHOC_SCENARIO}.csv", index=False)
    results[model] = res_m
    print(f"modèle ({model}) : {len(res_m)} lignes (blocs) et {len(res_s)} lignes (secteurs sommés) écrites dans {out}")


In [ ]:
# moyenne par bloc acheteur (sur tous les secteurs) : -0.01 pour dlnCost = coût unitaire inférieur d'environ 1 % à la baseline
res = results[2]
cols = [c_ for c_ in ("dlnS_D", "dlnS_F", "dlnCost") if c_ in res.columns]
res.groupby(["scenario", "bloc"])[cols].mean().round(4)
